In [ ]:
"""
Analyze per-pair min/max statistics of the real LR depth vs. the HR GT depth
for the TOFDSR dataset, and quantify how often the HR GT falls outside the
LR min/max range (i.e., pixels that would be clipped when the HR target is
normalized with LR statistics).

Usage:
    python analyze_tofdsr_minmax.py --base /path/to/TOFDSR \
        --split train --low 0.05 --high 5.0
    python analyze_tofdsr_minmax.py --base /path/to/TOFDSR \
        --split test --low 0.1 --high 6.0

Outputs a CSV (one row per pair) and prints a summary.
"""

import argparse
import csv
import os

import numpy as np
from PIL import Image


def get_pairs(list_file: str, base: str):
    """Mirror ProcessingTOFDSRReal._GetPairs: (hr_gt, rgb, lr) per line."""
    pairs = []
    with open(list_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            # parts[0]=rgb, parts[1]=hr gt depth, parts[2]=real lr depth
            # NOTE: original code uses lstrip('TOFDC_split') which strips a
            # character SET. removeprefix is the safe equivalent.
            hr = base + parts[1].removeprefix("TOFDC_split")
            rgb = base + parts[0].removeprefix("TOFDC_split")
            lr = base + parts[2].removeprefix("TOFDC_split")
            pairs.append((hr, rgb, lr))
    return pairs


def load_depth_m(path: str) -> np.ndarray:
    """Load a depth PNG stored in millimeters, return float32 meters."""
    return np.asarray(Image.open(path), dtype=np.float32) / 1000.0


def analyze(pairs, low: float, high: float, out_csv: str):
    rows = []
    for idx, (hr_path, _rgb_path, lr_path) in enumerate(pairs):
        hr = np.clip(load_depth_m(hr_path), low, high)
        lr = np.clip(load_depth_m(lr_path), low, high)

        hr_min, hr_max = float(hr.min()), float(hr.max())
        lr_min, lr_max = float(lr.min()), float(lr.max())

        # Differences (positive max_diff => HR exceeds LR range at the top;
        # negative min_diff => HR goes below the LR range at the bottom).
        max_diff = hr_max - lr_max
        min_diff = hr_min - lr_min

        # Pixels of the HR GT that would be clipped when normalized with
        # the LR min/max (exactly what _NormalizeDepthWithMinMax clips).
        n_pix = hr.size
        frac_above = float((hr > lr_max).sum()) / n_pix
        frac_below = float((hr < lr_min).sum()) / n_pix

        # Worst-case error introduced by that clipping, in meters.
        overshoot = max(hr_max - lr_max, 0.0)
        undershoot = max(lr_min - hr_min, 0.0)

        rows.append(
            {
                "index": idx,
                "hr_path": os.path.basename(hr_path),
                "lr_min": lr_min,
                "lr_max": lr_max,
                "hr_min": hr_min,
                "hr_max": hr_max,
                "min_diff_hr_minus_lr": min_diff,
                "max_diff_hr_minus_lr": max_diff,
                "lr_range": lr_max - lr_min,
                "hr_range": hr_max - hr_min,
                "frac_hr_above_lr_max": frac_above,
                "frac_hr_below_lr_min": frac_below,
                "clip_overshoot_m": overshoot,
                "clip_undershoot_m": undershoot,
            }
        )

        if (idx + 1) % 100 == 0:
            print(f"  processed {idx + 1}/{len(pairs)}")

    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    return rows


def summarize(rows, low, high):
    def col(name):
        return np.array([r[name] for r in rows], dtype=np.float64)

    max_diff = col("max_diff_hr_minus_lr")
    min_diff = col("min_diff_hr_minus_lr")
    frac_above = col("frac_hr_above_lr_max")
    frac_below = col("frac_hr_below_lr_min")
    over = col("clip_overshoot_m")
    under = col("clip_undershoot_m")

    n = len(rows)
    print(f"\n===== Summary over {n} pairs (clip range [{low}, {high}] m) =====")
    print(f"max_diff (hr_max - lr_max):  mean {max_diff.mean():+.4f} m | "
          f"median {np.median(max_diff):+.4f} | min {max_diff.min():+.4f} | "
          f"max {max_diff.max():+.4f}")
    print(f"min_diff (hr_min - lr_min):  mean {min_diff.mean():+.4f} m | "
          f"median {np.median(min_diff):+.4f} | min {min_diff.min():+.4f} | "
          f"max {min_diff.max():+.4f}")
    print(f"pairs where HR exceeds LR max: {(max_diff > 0).sum()} "
          f"({100.0 * (max_diff > 0).mean():.1f}%)")
    print(f"pairs where HR goes below LR min: {(min_diff < 0).sum()} "
          f"({100.0 * (min_diff < 0).mean():.1f}%)")
    print(f"HR pixels clipped above (per pair): mean {100 * frac_above.mean():.3f}% | "
          f"worst {100 * frac_above.max():.3f}%")
    print(f"HR pixels clipped below (per pair): mean {100 * frac_below.mean():.3f}% | "
          f"worst {100 * frac_below.max():.3f}%")
    print(f"worst-case clipping error: overshoot {over.max():.4f} m, "
          f"undershoot {under.max():.4f} m")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True,
                    help="Path to the TOFDSR folder (no trailing slash)")
    ap.add_argument("--split", choices=["train", "test"], default="train")
    ap.add_argument("--low", type=float, default=None,
                    help="Clip low (defaults: train 0.05, test 0.1)")
    ap.add_argument("--high", type=float, default=None,
                    help="Clip high (defaults: train 5.0, test 6.0)")
    ap.add_argument("--out", default=None, help="Output CSV path")
    args = ap.parse_args()

    low = args.low if args.low is not None else (0.05 if args.split == "train" else 0.1)
    high = args.high if args.high is not None else (5.0 if args.split == "train" else 6.0)

    list_file = os.path.join(
        args.base, f"TOFDSR_{'Train' if args.split == 'train' else 'Test'}.txt")
    out_csv = args.out or f"tofdsr_minmax_{args.split}.csv"

    pairs = get_pairs(list_file, args.base)
    print(f"{len(pairs)} pairs in {args.split} split; clip [{low}, {high}] m")

    rows = analyze(pairs, low, high, out_csv)
    summarize(rows, low, high)
    print(f"\nPer-pair results written to {out_csv}")




In [ ]:
main()